
# Đánh giá mô hình RVC / Voice Conversion bằng dữ liệu giọng nói sạch

Notebook này hỗ trợ chuẩn bị dữ liệu và đánh giá mô hình chuyển đổi giọng nói ở phạm vi **converted voice** trước khi mixing với nhạc nền.

Ba nhóm chỉ số được triển khai:

1. **Speaker Similarity**: đo độ giống giữa giọng đã chuyển đổi và giọng mục tiêu bằng speaker embedding cosine similarity.
2. **Predicted MOS / Naturalness**: ước lượng độ tự nhiên/chất lượng âm thanh bằng mô hình dự đoán MOS nếu môi trường cài được backend phù hợp.
3. **ASR WER/CER**: đo khả năng giữ nội dung câu nói bằng nhận dạng giọng nói tự động và so sánh transcript.

Notebook được thiết kế cho dữ liệu LibriTTS/OpenSLR, nhưng có thể dùng với dữ liệu khác nếu bạn sắp xếp theo cấu trúc tương tự.



## Cách sử dụng nhanh

1. Tải và giải nén `train-clean-100.tar.gz` và `test-clean.tar.gz` của LibriTTS.
2. Sửa biến `DATASET_ROOT` trong phần cấu hình để trỏ đến thư mục `LibriTTS`.
3. Chạy lần lượt các cell đến phần **Chuẩn bị dữ liệu đánh giá**.
4. Dùng thư mục `prepared_data/target_train` để train mô hình RVC.
5. Dùng các file trong `prepared_data/source_test` để chạy inference bằng mô hình RVC đã train.
6. Đặt kết quả converted vào thư mục:

```text
rvc_eval_project/converted_output/<ten_model>/source_001.wav
rvc_eval_project/converted_output/<ten_model>/source_002.wav
...
```

7. Chạy tiếp các cell đánh giá để xuất file kết quả `.csv` và bảng tổng hợp.

> Lưu ý: notebook này không phụ thuộc vào một repo RVC cụ thể. Phần train/inference RVC có thể chạy bằng hệ thống hiện có của bạn. Notebook này chuẩn bị dữ liệu, kiểm tra output và đánh giá định lượng.


## 1. Cài đặt thư viện

In [ ]:

# =========================
# 1. Cài thư viện cần thiết
# =========================
# Chạy cell này nếu môi trường của bạn chưa cài đủ package.
# Nếu đang dùng môi trường đã có sẵn torch/torchaudio, quá trình cài sẽ nhanh hơn.

RUN_INSTALL = True
INSTALL_OPTIONAL_MOS_BACKEND = False  # Bật True nếu muốn thử cài backend dự đoán MOS qua torch.hub/git.

if RUN_INSTALL:
    import importlib.util
    import subprocess
    import sys

    packages = {
        "numpy": "numpy",
        "pandas": "pandas",
        "tqdm": "tqdm",
        "soundfile": "soundfile",
        "librosa": "librosa",
        "sklearn": "scikit-learn",
        "matplotlib": "matplotlib",
        "jiwer": "jiwer",
        "transformers": "transformers",
        "speechbrain": "speechbrain",
        # torch và torchaudio thường đã có sẵn trên Colab/Kaggle.
        # Nếu thiếu, bạn có thể tự cài theo môi trường CUDA/CPU của máy.
    }

    for import_name, pip_name in packages.items():
        if importlib.util.find_spec(import_name) is None:
            print(f"Installing {pip_name}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        else:
            print(f"OK: {import_name}")

    if INSTALL_OPTIONAL_MOS_BACKEND:
        # Backend UTMOS/SpeechMOS có thể thay đổi theo thời gian.
        # Nếu cell MOS phía dưới không chạy được, bạn vẫn có thể bỏ qua predicted MOS
        # và chỉ dùng Speaker Similarity + WER/CER.
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/tarepan/SpeechMOS.git"])
        except Exception as e:
            print("Không cài được SpeechMOS backend. Vẫn có thể chạy các chỉ số còn lại.", e)


In [ ]:

# =========================
# 2. Import và cấu hình
# =========================
from pathlib import Path
import os
import re
import tarfile
import shutil
import random
import math
import json
import subprocess
from typing import List, Optional, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import soundfile as sf
import librosa

import torch
import torchaudio
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

# -------------------------
# CẤU HÌNH ĐƯỜNG DẪN
# -------------------------
# Thư mục gốc chứa LibriTTS sau khi giải nén.
# Ví dụ:
#   /content/LibriTTS
#   D:/datasets/LibriTTS
#   ./LibriTTS
DATASET_ROOT = Path("./rvc_eval_project/raw_data/LibriTTS")

# Nếu bạn chỉ mới tải file .tar.gz và chưa giải nén, đặt ARCHIVE_DIR là nơi chứa file .tar.gz.
ARCHIVE_DIR = Path("./downloads")

# Thư mục làm việc của notebook.
PROJECT_DIR = Path("./rvc_eval_project")
PREPARED_DIR = PROJECT_DIR / "prepared_data"
CONVERTED_ROOT = PROJECT_DIR / "converted_output"
RESULTS_DIR = PROJECT_DIR / "results"

# -------------------------
# CẤU HÌNH DATASET
# -------------------------
TARGET_SPLIT = "train-clean-100"
SOURCE_SPLIT = "test-clean"

# Để None thì notebook tự chọn speaker có đủ dữ liệu.
TARGET_SPEAKER_ID = None
SOURCE_SPEAKER_ID = None

# Số lượng dữ liệu cần tách từ target speaker.
TARGET_TRAIN_SECONDS = 10 * 60     # 10 phút để train RVC
TARGET_EVAL_SECONDS = 2 * 60       # 2 phút giữ lại để đánh giá speaker similarity

# Số câu source test dùng để chạy RVC inference và tính WER/CER.
SOURCE_TEST_COUNT = 10
MIN_SOURCE_DURATION = 2.0
MAX_SOURCE_DURATION = 12.0

# Chọn có chuyển audio sang WAV khi chuẩn bị dữ liệu hay không.
# RVC thường nhận WAV tốt hơn, nên mặc định chuyển sang WAV mono nhưng giữ sample rate gốc.
PREPARED_AUDIO_SR = None  # None = giữ sample rate gốc; ví dụ 16000 = ép về 16 kHz.

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -------------------------
# CẤU HÌNH ASR
# -------------------------
# Với LibriTTS tiếng Anh: dùng english.
# Với dữ liệu tiếng Việt: đổi thành "vietnamese" hoặc để None cho Whisper tự nhận diện.
ASR_MODEL_ID = "openai/whisper-small"
ASR_LANGUAGE = "english"

# -------------------------
# CẤU HÌNH MOS / NATURALNESS
# -------------------------
RUN_PREDICTED_MOS = True
MOS_BACKEND = "utmos_torchhub"  # hiện hỗ trợ thử nghiệm: utmos_torchhub; nếu lỗi sẽ ghi NaN.

# Tạo thư mục.
for p in [PROJECT_DIR, PREPARED_DIR, CONVERTED_ROOT, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR.resolve())
print("DATASET_ROOT:", DATASET_ROOT.resolve())


In [ ]:

# =====================================
# 3. Tải / giải nén LibriTTS nếu cần
# =====================================
# Mặc định không tự tải vì train-clean-100 khá lớn (~7.7GB).
# Nếu muốn tự tải bằng notebook, đặt AUTO_DOWNLOAD=True.

AUTO_DOWNLOAD = False
AUTO_EXTRACT = False

LIBRITTS_URLS = {
    "test-clean": "https://www.openslr.org/resources/60/test-clean.tar.gz",
    "train-clean-100": "https://www.openslr.org/resources/60/train-clean-100.tar.gz",
}

SPLITS_TO_CHECK = [TARGET_SPLIT, SOURCE_SPLIT]


def find_split_dir(dataset_root: Path, split_name: str) -> Optional[Path]:
    """Tìm thư mục split trong DATASET_ROOT, kể cả khi DATASET_ROOT đặt lệch 1 cấp."""
    candidates = [
        dataset_root / split_name,
        dataset_root / "LibriTTS" / split_name,
        dataset_root.parent / "LibriTTS" / split_name,
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    # Tìm sâu hơn nếu cần.
    if dataset_root.exists():
        for p in dataset_root.rglob(split_name):
            if p.is_dir():
                return p
    return None


def download_file(url: str, dest: Path):
    import urllib.request
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Đã có file: {dest}")
        return
    print(f"Đang tải {url} -> {dest}")
    urllib.request.urlretrieve(url, dest)


def extract_archive(archive_path: Path, extract_to: Path):
    print(f"Đang giải nén {archive_path} -> {extract_to}")
    extract_to.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=extract_to)

# Kiểm tra dữ liệu đã có chưa.
for split in SPLITS_TO_CHECK:
    split_dir = find_split_dir(DATASET_ROOT, split)
    print(f"Split {split}:", "FOUND" if split_dir else "MISSING", split_dir if split_dir else "")

# Tự tải nếu bật.
if AUTO_DOWNLOAD:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    for split in SPLITS_TO_CHECK:
        if find_split_dir(DATASET_ROOT, split) is None:
            url = LIBRITTS_URLS[split]
            archive_path = ARCHIVE_DIR / f"{split}.tar.gz"
            download_file(url, archive_path)

# Tự giải nén nếu có archive.
if AUTO_EXTRACT:
    # Giải nén vào DATASET_ROOT.parent để archive tạo thư mục LibriTTS/ đúng cấu trúc.
    extract_to = DATASET_ROOT.parent
    for split in SPLITS_TO_CHECK:
        if find_split_dir(DATASET_ROOT, split) is None:
            archive_path = ARCHIVE_DIR / f"{split}.tar.gz"
            if archive_path.exists():
                extract_archive(archive_path, extract_to)
            else:
                print(f"Không tìm thấy archive cho {split}: {archive_path}")

# Kiểm tra lại sau tải/giải nén.
print("\nKết quả sau kiểm tra/tải/giải nén:")
for split in SPLITS_TO_CHECK:
    split_dir = find_split_dir(DATASET_ROOT, split)
    print(f"Split {split}:", "FOUND" if split_dir else "MISSING", split_dir if split_dir else "")


In [ ]:

# =========================================
# 4. Quét metadata audio và transcript
# =========================================
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}


def read_audio_duration(path: Path) -> float:
    """Đọc duration audio bằng soundfile, fallback sang torchaudio."""
    try:
        info = sf.info(str(path))
        return float(info.frames) / float(info.samplerate)
    except Exception:
        try:
            info = torchaudio.info(str(path))
            return float(info.num_frames) / float(info.sample_rate)
        except Exception:
            return np.nan


def read_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8").strip()
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1").strip()


def get_transcript_for_audio(audio_path: Path) -> str:
    """Tìm transcript cho LibriTTS/LibriSpeech hoặc dữ liệu có file .txt đi kèm."""
    # LibriTTS thường có file .normalized.txt hoặc .original.txt cùng stem.
    for suffix in [".normalized.txt", ".original.txt", ".txt"]:
        cand = audio_path.with_suffix(suffix)
        if cand.exists():
            txt = read_text_file(cand)
            if txt:
                return txt

    # LibriSpeech thường có file *.trans.txt trong cùng folder: <utt_id> transcript
    utt_id = audio_path.stem
    for trans_file in audio_path.parent.glob("*.trans.txt"):
        try:
            for line in read_text_file(trans_file).splitlines():
                if line.startswith(utt_id + " "):
                    return line[len(utt_id) + 1 :].strip()
        except Exception:
            pass

    # Một số dữ liệu có .tsv.
    for tsv_file in audio_path.parent.glob("*.tsv"):
        try:
            df = pd.read_csv(tsv_file, sep="\t")
            for col_id in ["id", "file", "path", "utt_id"]:
                if col_id in df.columns:
                    match = df[df[col_id].astype(str).str.contains(utt_id, regex=False)]
                    if not match.empty:
                        for col_txt in ["transcript", "text", "sentence", "normalized_text"]:
                            if col_txt in match.columns:
                                return str(match.iloc[0][col_txt])
        except Exception:
            pass

    return ""


def infer_speaker_id(audio_path: Path, split_dir: Path) -> str:
    """Với LibriTTS: split/speaker/chapter/file.wav -> speaker."""
    try:
        rel = audio_path.relative_to(split_dir)
        if len(rel.parts) >= 3:
            return str(rel.parts[0])
    except Exception:
        pass
    # fallback: dùng folder cha gần nhất.
    return audio_path.parent.name


def scan_split(split_name: str) -> pd.DataFrame:
    split_dir = find_split_dir(DATASET_ROOT, split_name)
    if split_dir is None:
        raise FileNotFoundError(f"Không tìm thấy split {split_name}. Hãy kiểm tra DATASET_ROOT hoặc giải nén dữ liệu.")

    audio_files = [p for p in split_dir.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS]
    rows = []
    for p in tqdm(audio_files, desc=f"Scanning {split_name}"):
        duration = read_audio_duration(p)
        rows.append({
            "split": split_name,
            "speaker_id": infer_speaker_id(p, split_dir),
            "path": str(p),
            "file_name": p.name,
            "stem": p.stem,
            "duration_sec": duration,
            "transcript": get_transcript_for_audio(p),
        })
    return pd.DataFrame(rows)

train_df = scan_split(TARGET_SPLIT)
source_df = scan_split(SOURCE_SPLIT)

print("train_df:", train_df.shape)
print("source_df:", source_df.shape)

display(train_df.head())
display(source_df.head())

# Lưu metadata để kiểm tra lại.
train_df.to_csv(RESULTS_DIR / f"metadata_{TARGET_SPLIT}.csv", index=False, encoding="utf-8-sig")
source_df.to_csv(RESULTS_DIR / f"metadata_{SOURCE_SPLIT}.csv", index=False, encoding="utf-8-sig")


In [ ]:

# =======================================
# 5. Thống kê speaker và tự chọn speaker
# =======================================
def speaker_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = (
        df.groupby("speaker_id")
        .agg(
            n_files=("path", "count"),
            total_sec=("duration_sec", "sum"),
            avg_sec=("duration_sec", "mean"),
            n_with_transcript=("transcript", lambda x: sum(bool(str(v).strip()) for v in x)),
        )
        .reset_index()
    )
    out["total_min"] = out["total_sec"] / 60.0
    return out.sort_values("total_sec", ascending=False)

train_spk = speaker_summary(train_df)
source_spk = speaker_summary(source_df)

print("Top target speakers:")
display(train_spk.head(10))
print("Top source speakers:")
display(source_spk.head(10))

needed_target_sec = TARGET_TRAIN_SECONDS + TARGET_EVAL_SECONDS

if TARGET_SPEAKER_ID is None:
    candidates = train_spk[train_spk["total_sec"] >= needed_target_sec]
    if candidates.empty:
        raise ValueError(f"Không có speaker đủ {needed_target_sec/60:.1f} phút trong {TARGET_SPLIT}.")
    TARGET_SPEAKER_ID = str(candidates.iloc[0]["speaker_id"])

if SOURCE_SPEAKER_ID is None:
    candidates = source_spk[source_spk["n_with_transcript"] >= SOURCE_TEST_COUNT]
    # Nếu trùng speaker_id với target thì chọn người khác nếu có.
    candidates2 = candidates[candidates["speaker_id"].astype(str) != str(TARGET_SPEAKER_ID)]
    if not candidates2.empty:
        candidates = candidates2
    if candidates.empty:
        raise ValueError(f"Không có source speaker đủ {SOURCE_TEST_COUNT} câu có transcript trong {SOURCE_SPLIT}.")
    SOURCE_SPEAKER_ID = str(candidates.iloc[0]["speaker_id"])

print("TARGET_SPEAKER_ID:", TARGET_SPEAKER_ID)
print("SOURCE_SPEAKER_ID:", SOURCE_SPEAKER_ID)


In [ ]:

# ==================================================
# 6. Chọn file và chuẩn bị thư mục đánh giá
# ==================================================
def select_by_duration(df: pd.DataFrame, total_seconds: float, seed: int = 42) -> pd.DataFrame:
    d = df.dropna(subset=["duration_sec"]).copy()
    d = d[d["duration_sec"] > 0]
    d = d.sample(frac=1, random_state=seed).reset_index(drop=True)
    selected = []
    acc = 0.0
    for _, row in d.iterrows():
        selected.append(row)
        acc += float(row["duration_sec"])
        if acc >= total_seconds:
            break
    return pd.DataFrame(selected)


def convert_to_wav(src: Path, dst: Path, sr: Optional[int] = None):
    """Convert audio sang WAV mono. sr=None để giữ sample rate gốc."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    y, orig_sr = librosa.load(str(src), sr=sr, mono=True)
    out_sr = sr if sr is not None else orig_sr
    # Tránh clipping do scale bất thường.
    if np.max(np.abs(y)) > 1.0:
        y = y / np.max(np.abs(y))
    sf.write(str(dst), y, out_sr)


def prepare_audio_set(df: pd.DataFrame, output_dir: Path, prefix: str, sr: Optional[int] = None) -> pd.DataFrame:
    rows = []
    output_dir.mkdir(parents=True, exist_ok=True)
    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"Preparing {prefix}"), start=1):
        src = Path(row["path"])
        out_name = f"{prefix}_{idx:03d}.wav"
        dst = output_dir / out_name
        convert_to_wav(src, dst, sr=sr)
        rows.append({
            "id": f"{prefix}_{idx:03d}",
            "file": out_name,
            "path": str(dst),
            "original_path": str(src),
            "speaker_id": row["speaker_id"],
            "duration_sec": read_audio_duration(dst),
            "transcript": row.get("transcript", ""),
        })
    return pd.DataFrame(rows)

# Xóa dữ liệu chuẩn bị cũ nếu muốn làm lại.
RESET_PREPARED_DATA = True
if RESET_PREPARED_DATA and PREPARED_DIR.exists():
    shutil.rmtree(PREPARED_DIR)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

# Chọn target train/eval từ cùng speaker nhưng không trùng file.
target_all = train_df[train_df["speaker_id"].astype(str) == str(TARGET_SPEAKER_ID)].copy()
target_all = target_all.dropna(subset=["duration_sec"])
target_all = target_all[target_all["duration_sec"] > 0].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

target_train_rows = []
target_eval_rows = []
acc = 0.0
used_indices = set()
for i, row in target_all.iterrows():
    if acc < TARGET_TRAIN_SECONDS:
        target_train_rows.append(row)
        used_indices.add(i)
        acc += float(row["duration_sec"])

acc_eval = 0.0
for i, row in target_all.iterrows():
    if i in used_indices:
        continue
    if acc_eval < TARGET_EVAL_SECONDS:
        target_eval_rows.append(row)
        acc_eval += float(row["duration_sec"])

target_train_sel = pd.DataFrame(target_train_rows)
target_eval_sel = pd.DataFrame(target_eval_rows)

# Chọn source test từ speaker khác, có transcript, duration hợp lý.
source_pool = source_df[source_df["speaker_id"].astype(str) == str(SOURCE_SPEAKER_ID)].copy()
source_pool = source_pool[source_pool["transcript"].astype(str).str.len() > 0]
source_pool = source_pool[source_pool["duration_sec"].between(MIN_SOURCE_DURATION, MAX_SOURCE_DURATION)]
if len(source_pool) < SOURCE_TEST_COUNT:
    print("Cảnh báo: không đủ source theo điều kiện duration/transcript. Sẽ nới điều kiện duration.")
    source_pool = source_df[source_df["speaker_id"].astype(str) == str(SOURCE_SPEAKER_ID)].copy()
    source_pool = source_pool[source_pool["transcript"].astype(str).str.len() > 0]

source_sel = source_pool.sample(n=min(SOURCE_TEST_COUNT, len(source_pool)), random_state=RANDOM_SEED).reset_index(drop=True)

print("Target train minutes:", target_train_sel["duration_sec"].sum()/60)
print("Target eval minutes:", target_eval_sel["duration_sec"].sum()/60)
print("Source test files:", len(source_sel))

# Chuẩn bị folder.
target_train_prepared = prepare_audio_set(target_train_sel, PREPARED_DIR / "target_train", "target_train", PREPARED_AUDIO_SR)
target_eval_prepared = prepare_audio_set(target_eval_sel, PREPARED_DIR / "target_eval", "target_eval", PREPARED_AUDIO_SR)
source_test_prepared = prepare_audio_set(source_sel, PREPARED_DIR / "source_test", "source", PREPARED_AUDIO_SR)

# Lưu manifest.
target_train_prepared.to_csv(PREPARED_DIR / "target_train_manifest.csv", index=False, encoding="utf-8-sig")
target_eval_prepared.to_csv(PREPARED_DIR / "target_eval_manifest.csv", index=False, encoding="utf-8-sig")
source_test_prepared.to_csv(PREPARED_DIR / "source_test_manifest.csv", index=False, encoding="utf-8-sig")
source_test_prepared[["file", "transcript"]].to_csv(PREPARED_DIR / "transcript.csv", index=False, encoding="utf-8-sig")

print("Đã chuẩn bị dữ liệu tại:", PREPARED_DIR.resolve())
display(source_test_prepared[["file", "duration_sec", "transcript"]])



## 7. Train và inference RVC

Sau cell chuẩn bị dữ liệu, bạn cần thực hiện phần RVC như sau:

- Dùng thư mục `prepared_data/target_train` để train model RVC.
- Sau khi có model RVC, dùng từng file trong `prepared_data/source_test` để chuyển đổi sang giọng mục tiêu.
- Đặt kết quả converted vào thư mục `converted_output/<ten_model>/`.

Tên file output nên giữ đúng tên file source:

```text
source_001.wav -> converted_output/model_10min/source_001.wav
source_002.wav -> converted_output/model_10min/source_002.wav
...
```

Nếu muốn so sánh nhiều model, tạo nhiều thư mục:

```text
converted_output/model_5min/
converted_output/model_10min/
converted_output/model_15min/
```


In [ ]:

# =============================================================
# 8. Tùy chọn: chạy RVC inference bằng command template nếu có
# =============================================================
# Nếu bạn có command inference của repo RVC, điền template dưới đây.
# Các placeholder dùng được: {input}, {output}, {model_name}
# Ví dụ giả định:
# RVC_INFER_COMMAND_TEMPLATE = 'python infer.py --input "{input}" --output "{output}" --model my_model.pth --index my_model.index'

RUN_RVC_INFERENCE_FROM_NOTEBOOK = False
MODEL_NAME_FOR_INFERENCE = "model_default"
RVC_INFER_COMMAND_TEMPLATE = ""


def build_inference_todo(model_name: str) -> pd.DataFrame:
    source_manifest = pd.read_csv(PREPARED_DIR / "source_test_manifest.csv")
    out_dir = CONVERTED_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for _, row in source_manifest.iterrows():
        rows.append({
            "input": row["path"],
            "output": str(out_dir / row["file"]),
            "model_name": model_name,
            "transcript": row["transcript"],
        })
    return pd.DataFrame(rows)

inference_todo = build_inference_todo(MODEL_NAME_FOR_INFERENCE)
inference_todo.to_csv(PROJECT_DIR / "rvc_inference_todo.csv", index=False, encoding="utf-8-sig")
print("Đã tạo danh sách file cần inference:", (PROJECT_DIR / "rvc_inference_todo.csv").resolve())
display(inference_todo.head())

if RUN_RVC_INFERENCE_FROM_NOTEBOOK:
    if not RVC_INFER_COMMAND_TEMPLATE.strip():
        raise ValueError("Bạn cần điền RVC_INFER_COMMAND_TEMPLATE trước khi chạy inference.")
    for _, row in tqdm(inference_todo.iterrows(), total=len(inference_todo), desc="RVC inference"):
        cmd = RVC_INFER_COMMAND_TEMPLATE.format(
            input=row["input"],
            output=row["output"],
            model_name=row["model_name"],
        )
        print("RUN:", cmd)
        subprocess.run(cmd, shell=True, check=True)
else:
    print("Chưa chạy RVC inference từ notebook.")
    print("Hãy train/infer bằng pipeline RVC của bạn, rồi đặt file output vào:")
    print((CONVERTED_ROOT / MODEL_NAME_FOR_INFERENCE).resolve())


In [ ]:

# ===========================================
# 9. Kiểm tra converted_output đã có chưa
# ===========================================
def get_model_dirs(converted_root: Path) -> List[Path]:
    if not converted_root.exists():
        return []
    return sorted([p for p in converted_root.iterdir() if p.is_dir()])


def check_converted_outputs() -> pd.DataFrame:
    source_manifest = pd.read_csv(PREPARED_DIR / "source_test_manifest.csv")
    model_dirs = get_model_dirs(CONVERTED_ROOT)
    rows = []
    for model_dir in model_dirs:
        for _, row in source_manifest.iterrows():
            expected = model_dir / row["file"]
            rows.append({
                "model": model_dir.name,
                "source_file": row["file"],
                "converted_path": str(expected),
                "exists": expected.exists(),
            })
    return pd.DataFrame(rows)

converted_check = check_converted_outputs()
if converted_check.empty:
    print("Chưa có thư mục model nào trong converted_output.")
    print("Hãy tạo ví dụ:", (CONVERTED_ROOT / "model_10min").resolve())
else:
    display(converted_check)
    print(converted_check.groupby("model")["exists"].agg(["sum", "count"]))

# Nếu chưa có đủ output, các cell đánh giá phía dưới sẽ bỏ qua file thiếu và cảnh báo.


In [ ]:

# =====================================================
# 10. Đánh giá Speaker Similarity bằng SpeechBrain ECAPA
# =====================================================
from speechbrain.inference.speaker import EncoderClassifier

SPEAKER_MODEL_SOURCE = "speechbrain/spkrec-ecapa-voxceleb"
SPEAKER_MODEL_DIR = str(PROJECT_DIR / "pretrained_models" / "spkrec-ecapa-voxceleb")

print("Đang load speaker embedding model...")
speaker_encoder = EncoderClassifier.from_hparams(
    source=SPEAKER_MODEL_SOURCE,
    savedir=SPEAKER_MODEL_DIR,
    run_opts={"device": DEVICE},
)
print("Loaded speaker model")


def load_audio_16k(path: Path) -> torch.Tensor:
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    wav = wav.squeeze(0)
    # Chuẩn hóa biên độ nhẹ.
    if wav.abs().max() > 0:
        wav = wav / wav.abs().max()
    return wav


def speaker_embedding(path: Path) -> np.ndarray:
    wav = load_audio_16k(path).to(DEVICE)
    with torch.no_grad():
        emb = speaker_encoder.encode_batch(wav.unsqueeze(0))
    emb = emb.squeeze().detach().cpu().numpy().astype(np.float32)
    return emb


def mean_speaker_embedding(paths: List[Path]) -> np.ndarray:
    embs = []
    for p in tqdm(paths, desc="Embedding target_eval"):
        embs.append(speaker_embedding(p))
    return np.mean(np.vstack(embs), axis=0)


def cos_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = a.reshape(1, -1)
    b = b.reshape(1, -1)
    return float(cosine_similarity(a, b)[0, 0])

# Target embedding từ target_eval.
target_eval_manifest = pd.read_csv(PREPARED_DIR / "target_eval_manifest.csv")
source_manifest = pd.read_csv(PREPARED_DIR / "source_test_manifest.csv")
target_eval_paths = [Path(p) for p in target_eval_manifest["path"].tolist()]
target_centroid = mean_speaker_embedding(target_eval_paths)

# Embedding source baseline.
source_embeddings = {}
for _, row in tqdm(source_manifest.iterrows(), total=len(source_manifest), desc="Embedding source_test"):
    source_embeddings[row["file"]] = speaker_embedding(Path(row["path"]))

# Đánh giá từng model trong converted_output.
sim_rows = []
for model_dir in get_model_dirs(CONVERTED_ROOT):
    for _, row in tqdm(source_manifest.iterrows(), total=len(source_manifest), desc=f"Speaker sim {model_dir.name}"):
        converted_path = model_dir / row["file"]
        if not converted_path.exists():
            sim_rows.append({
                "model": model_dir.name,
                "file": row["file"],
                "source_target_similarity": cos_sim(source_embeddings[row["file"]], target_centroid),
                "converted_target_similarity": np.nan,
                "improvement": np.nan,
                "converted_exists": False,
            })
            continue
        converted_emb = speaker_embedding(converted_path)
        src_tgt = cos_sim(source_embeddings[row["file"]], target_centroid)
        conv_tgt = cos_sim(converted_emb, target_centroid)
        sim_rows.append({
            "model": model_dir.name,
            "file": row["file"],
            "source_target_similarity": src_tgt,
            "converted_target_similarity": conv_tgt,
            "improvement": conv_tgt - src_tgt,
            "converted_exists": True,
        })

speaker_similarity_df = pd.DataFrame(sim_rows)
speaker_similarity_df.to_csv(RESULTS_DIR / "speaker_similarity_results.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", RESULTS_DIR / "speaker_similarity_results.csv")
display(speaker_similarity_df)

if not speaker_similarity_df.empty:
    display(
        speaker_similarity_df.groupby("model")[["source_target_similarity", "converted_target_similarity", "improvement"]]
        .mean(numeric_only=True)
        .reset_index()
    )


In [ ]:

# ===================================================
# 11. Đánh giá Naturalness bằng Predicted MOS / UTMOS
# ===================================================
# Ghi chú:
# - MOSNet là hướng học thuật kinh điển cho VC, nhưng không phải lúc nào cũng dễ chạy trực tiếp.
# - Cell này thử dùng backend UTMOS qua torch.hub/SpeechMOS để dự đoán MOS.
# - Nếu backend không chạy được, notebook sẽ ghi NaN và bạn vẫn có thể dùng 2 metric còn lại.

mos_predictor = None
mos_backend_available = False


def try_load_utmos():
    """Thử load UTMOS qua torch.hub. API có thể thay đổi theo repo, nên có try/except."""
    try:
        predictor = torch.hub.load(
            "tarepan/SpeechMOS:v1.2.0",
            "utmos22_strong",
            trust_repo=True,
        )
        if hasattr(predictor, "to"):
            predictor = predictor.to(DEVICE)
        if hasattr(predictor, "eval"):
            predictor.eval()
        return predictor
    except Exception as e:
        print("Không load được UTMOS qua torch.hub:")
        print(type(e).__name__, e)
        return None


def predict_mos_utmos(path: Path) -> float:
    """Dự đoán MOS bằng UTMOS. Có nhiều fallback vì API có thể khác nhau theo phiên bản."""
    global mos_predictor
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav.to(DEVICE)
    with torch.no_grad():
        # Thử các kiểu gọi thường gặp.
        attempts = [
            lambda: mos_predictor(wav, sr),
            lambda: mos_predictor(wav.squeeze(0), sr),
            lambda: mos_predictor(wav.unsqueeze(0), sr),
            lambda: mos_predictor(str(path)),
        ]
        last_err = None
        for fn in attempts:
            try:
                score = fn()
                if isinstance(score, (tuple, list)):
                    score = score[0]
                if isinstance(score, torch.Tensor):
                    score = score.detach().cpu().numpy()
                score = np.asarray(score).reshape(-1)[0]
                return float(score)
            except Exception as e:
                last_err = e
        raise RuntimeError(f"Không predict được MOS cho {path}: {last_err}")

mos_rows = []
if RUN_PREDICTED_MOS:
    if MOS_BACKEND == "utmos_torchhub":
        mos_predictor = try_load_utmos()
        mos_backend_available = mos_predictor is not None
    else:
        print("MOS_BACKEND chưa hỗ trợ:", MOS_BACKEND)

    source_manifest = pd.read_csv(PREPARED_DIR / "source_test_manifest.csv")
    for model_dir in get_model_dirs(CONVERTED_ROOT):
        for _, row in tqdm(source_manifest.iterrows(), total=len(source_manifest), desc=f"MOS {model_dir.name}"):
            converted_path = model_dir / row["file"]
            if not converted_path.exists():
                mos_rows.append({"model": model_dir.name, "file": row["file"], "predicted_mos": np.nan, "mos_error": "missing converted"})
                continue
            if not mos_backend_available:
                mos_rows.append({"model": model_dir.name, "file": row["file"], "predicted_mos": np.nan, "mos_error": "MOS backend unavailable"})
                continue
            try:
                score = predict_mos_utmos(converted_path)
                mos_rows.append({"model": model_dir.name, "file": row["file"], "predicted_mos": score, "mos_error": ""})
            except Exception as e:
                mos_rows.append({"model": model_dir.name, "file": row["file"], "predicted_mos": np.nan, "mos_error": str(e)})
else:
    print("RUN_PREDICTED_MOS=False, bỏ qua Predicted MOS.")

mos_df = pd.DataFrame(mos_rows)
mos_df.to_csv(RESULTS_DIR / "predicted_mos_results.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", RESULTS_DIR / "predicted_mos_results.csv")
display(mos_df)

if not mos_df.empty:
    display(mos_df.groupby("model")[["predicted_mos"]].mean(numeric_only=True).reset_index())


In [ ]:

# ================================================
# 12. Đánh giá Content Preservation bằng ASR WER/CER
# ================================================
from transformers import pipeline
import jiwer
import unicodedata

print("Đang load ASR model:", ASR_MODEL_ID)
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=ASR_MODEL_ID,
    device=0 if DEVICE == "cuda" else -1,
)
print("Loaded ASR")


def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower().strip()
    # Chuẩn hóa unicode.
    s = unicodedata.normalize("NFC", s)
    # Xóa dấu câu nhưng giữ chữ và số unicode.
    s = re.sub(r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]", " ", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def transcribe_audio(path: Path) -> str:
    kwargs = {}
    if ASR_LANGUAGE:
        # Whisper pipeline nhận generate_kwargs cho language/task.
        kwargs["generate_kwargs"] = {"language": ASR_LANGUAGE, "task": "transcribe"}
    result = asr_pipe(str(path), **kwargs)
    if isinstance(result, dict):
        return result.get("text", "")
    return str(result)

asr_rows = []
source_manifest = pd.read_csv(PREPARED_DIR / "source_test_manifest.csv")
transcript_map = {row["file"]: row["transcript"] for _, row in source_manifest.iterrows()}

for model_dir in get_model_dirs(CONVERTED_ROOT):
    for _, row in tqdm(source_manifest.iterrows(), total=len(source_manifest), desc=f"ASR {model_dir.name}"):
        converted_path = model_dir / row["file"]
        ref = normalize_text(row["transcript"])
        if not converted_path.exists():
            asr_rows.append({
                "model": model_dir.name,
                "file": row["file"],
                "reference": ref,
                "hypothesis": "",
                "wer": np.nan,
                "cer": np.nan,
                "asr_error": "missing converted",
            })
            continue
        try:
            hyp_raw = transcribe_audio(converted_path)
            hyp = normalize_text(hyp_raw)
            wer_value = jiwer.wer(ref, hyp) if ref else np.nan
            cer_value = jiwer.cer(ref, hyp) if ref else np.nan
            asr_rows.append({
                "model": model_dir.name,
                "file": row["file"],
                "reference": ref,
                "hypothesis": hyp,
                "wer": wer_value,
                "cer": cer_value,
                "asr_error": "",
            })
        except Exception as e:
            asr_rows.append({
                "model": model_dir.name,
                "file": row["file"],
                "reference": ref,
                "hypothesis": "",
                "wer": np.nan,
                "cer": np.nan,
                "asr_error": str(e),
            })

asr_df = pd.DataFrame(asr_rows)
asr_df.to_csv(RESULTS_DIR / "asr_wer_cer_results.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", RESULTS_DIR / "asr_wer_cer_results.csv")
display(asr_df[["model", "file", "reference", "hypothesis", "wer", "cer", "asr_error"]])

if not asr_df.empty:
    display(asr_df.groupby("model")[["wer", "cer"]].mean(numeric_only=True).reset_index())


In [ ]:

# ==========================================
# 13. Gộp kết quả và tạo bảng tổng hợp cuối
# ==========================================
# Load lại từ file để cell này có thể chạy độc lập sau khi đánh giá xong.

def load_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

speaker_similarity_df = load_csv_if_exists(RESULTS_DIR / "speaker_similarity_results.csv")
mos_df = load_csv_if_exists(RESULTS_DIR / "predicted_mos_results.csv")
asr_df = load_csv_if_exists(RESULTS_DIR / "asr_wer_cer_results.csv")

# Gộp per-sample.
per_sample = None
if not speaker_similarity_df.empty:
    per_sample = speaker_similarity_df.copy()
if per_sample is not None and not mos_df.empty:
    per_sample = per_sample.merge(mos_df[["model", "file", "predicted_mos", "mos_error"]], on=["model", "file"], how="left")
if per_sample is not None and not asr_df.empty:
    per_sample = per_sample.merge(asr_df[["model", "file", "reference", "hypothesis", "wer", "cer", "asr_error"]], on=["model", "file"], how="left")

if per_sample is None:
    print("Chưa có kết quả để gộp. Hãy chạy các cell đánh giá trước.")
else:
    per_sample.to_csv(RESULTS_DIR / "per_sample_results.csv", index=False, encoding="utf-8-sig")
    display(per_sample)

    agg_dict = {
        "source_target_similarity": "mean",
        "converted_target_similarity": "mean",
        "improvement": "mean",
    }
    if "predicted_mos" in per_sample.columns:
        agg_dict["predicted_mos"] = "mean"
    if "wer" in per_sample.columns:
        agg_dict["wer"] = "mean"
    if "cer" in per_sample.columns:
        agg_dict["cer"] = "mean"

    summary = per_sample.groupby("model").agg(agg_dict).reset_index()
    summary["n_samples"] = per_sample.groupby("model")["file"].count().values

    # Sắp xếp ưu tiên: similarity cao, MOS cao, WER/CER thấp.
    sort_cols = []
    ascending = []
    if "converted_target_similarity" in summary.columns:
        sort_cols.append("converted_target_similarity")
        ascending.append(False)
    if "predicted_mos" in summary.columns:
        sort_cols.append("predicted_mos")
        ascending.append(False)
    if "wer" in summary.columns:
        sort_cols.append("wer")
        ascending.append(True)
    if sort_cols:
        summary = summary.sort_values(sort_cols, ascending=ascending)

    summary.to_csv(RESULTS_DIR / "summary_by_model.csv", index=False, encoding="utf-8-sig")
    print("Đã lưu:")
    print("-", RESULTS_DIR / "per_sample_results.csv")
    print("-", RESULTS_DIR / "summary_by_model.csv")
    display(summary)


In [ ]:

# ====================================
# 14. Vẽ biểu đồ kết quả tổng hợp
# ====================================
summary_path = RESULTS_DIR / "summary_by_model.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)

    metrics_to_plot = [
        ("converted_target_similarity", "Speaker similarity ↑"),
        ("predicted_mos", "Predicted MOS ↑"),
        ("wer", "WER ↓"),
        ("cer", "CER ↓"),
    ]

    for metric, title in metrics_to_plot:
        if metric not in summary.columns or summary[metric].isna().all():
            continue
        plt.figure(figsize=(8, 4))
        plt.bar(summary["model"], summary[metric])
        plt.title(title)
        plt.xlabel("Model")
        plt.ylabel(metric)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        out_path = RESULTS_DIR / f"plot_{metric}.png"
        plt.savefig(out_path, dpi=160)
        plt.show()
        print("Saved:", out_path)
else:
    print("Chưa có summary_by_model.csv")



## 15. Cách đọc kết quả

Bảng `summary_by_model.csv` là bảng chính để đưa vào báo cáo.

Ý nghĩa các cột:

| Cột | Ý nghĩa | Hướng tốt |
|---|---|---|
| `source_target_similarity` | Độ giống ban đầu giữa giọng nguồn và giọng mục tiêu | Chỉ là baseline |
| `converted_target_similarity` | Độ giống giữa giọng sau chuyển đổi và giọng mục tiêu | Càng cao càng tốt |
| `improvement` | Mức tăng similarity sau chuyển đổi so với source ban đầu | Càng cao càng tốt |
| `predicted_mos` | Điểm MOS dự đoán cho độ tự nhiên/chất lượng | Càng cao càng tốt |
| `wer` | Word Error Rate giữa transcript gốc và ASR của converted voice | Càng thấp càng tốt |
| `cer` | Character Error Rate giữa transcript gốc và ASR của converted voice | Càng thấp càng tốt |

Một mô hình có chất lượng tốt hơn khi:

```text
converted_target_similarity cao hơn
predicted_mos cao hơn
wer/cer thấp hơn
```

Không nên kết luận chỉ dựa vào một chỉ số duy nhất. Ví dụ, một mô hình có similarity cao nhưng WER/CER rất cao có nghĩa là giọng giống target hơn nhưng nội dung bị méo hoặc mất chữ.



## 16. Mẫu nhận xét đưa vào báo cáo

Sau khi có bảng kết quả, có thể viết nhận xét theo mẫu:

> Kết quả đánh giá cho thấy mô hình có độ tương đồng giọng nói sau chuyển đổi cao hơn so với độ tương đồng giữa giọng nguồn và giọng mục tiêu ban đầu. Điều này chứng tỏ mô hình đã học được một phần đặc trưng âm sắc của giọng mục tiêu. Bên cạnh đó, chỉ số WER/CER được sử dụng để kiểm tra khả năng giữ nội dung lời nói sau chuyển đổi. Mô hình có WER/CER thấp hơn cho thấy nội dung câu nói được bảo toàn tốt hơn. Đối với chỉ số predicted MOS, giá trị cao hơn phản ánh âm thanh đầu ra có xu hướng tự nhiên và ít méo hơn. Khi so sánh giữa các mô hình được huấn luyện với lượng dữ liệu khác nhau, mô hình có speaker similarity cao, predicted MOS cao và WER/CER thấp được xem là mô hình có chất lượng tốt hơn trong phạm vi đánh giá giọng nói sau chuyển đổi.

Tài liệu tham khảo gợi ý:

- VCC 2020 Objective Assessment: dùng ASV, speaker embeddings, predicted MOS và ASR để đánh giá khách quan hệ thống voice conversion.
- MOSNet: mô hình dự đoán MOS cho voice conversion.
- Speaker embedding / ASV: dùng để đo độ giống giọng mục tiêu.
- ASR WER/CER: dùng để kiểm tra khả năng giữ nội dung lời nói.
